<a href="https://colab.research.google.com/github/mihnea-popescu/yolov1-cupy/blob/avgpool2d/notebooks/avgpool2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git
%cd yolov1-cupy
#!git checkout avgpool2d

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 114 (delta 51), reused 83 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 2.96 MiB | 4.51 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/yolov1-cupy
Branch 'avgpool2d' set up to track remote branch 'avgpool2d' from 'origin'.
Switched to a new branch 'avgpool2d'


In [2]:
!pip install cupy-cuda12x
import cupy as cp
from avgpool2d import AvgPool2d

Test 1: Output shape is correct

In [3]:
pool = AvgPool2d(kernel_size=2, stride=2)
x = cp.random.randn(4, 3, 8, 8) # batch=4, channels=3, 8x8 image
out = pool(x)
assert out.shape == (4, 3, 4, 4), f"Expected (4,3,4,4), got {out.shape}"
print("Test 1 passed: output shape correct")

Test 1 passed: output shape correct


Test 2: Forward pass values are correct

In [4]:
pool = AvgPool2d(kernel_size=2, stride=2)

# 1 batch, 1 channel, 4x4 input — easy to verify by hand
x = cp.array([[[[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12],
                [13, 14, 15, 16]]]], dtype=cp.float32)

out = pool(x)

# Top-left window: avg(1,2,5,6) = 3.5
# Top-right window: avg(3,4,7,8) = 5.5
# Bottom-left window: avg(9,10,13,14) = 11.5
# Bottom-right window: avg(11,12,15,16) = 13.5
expected = cp.array([[[[3.5, 5.5],
                       [11.5, 13.5]]]], dtype=cp.float32)

cp.testing.assert_allclose(out, expected, rtol=1e-5)
print("Test 2 passed: forward pass values correct")

Test 2 passed: forward pass values correct


Test 3: Padding works correctly

In [5]:
pool = AvgPool2d(kernel_size=3, stride=1, padding=1)
x = cp.random.randn(2, 3, 6, 6)
out = pool(x)
# With padding=1, output should be same H and W as input
assert out.shape == (2, 3, 6, 6), f"Expected (2,3,6,6), got {out.shape}"
print("Test 3 passed: padding produces correct output shape")

Test 3 passed: padding produces correct output shape


Test 4: Backward pass gradient shape is correct

In [6]:
pool = AvgPool2d(kernel_size=2, stride=2)
x = cp.random.randn(2, 3, 8, 8)
out = pool(x)

d_out = cp.ones_like(out)
d_input = pool.backward(d_out)

assert d_input.shape == x.shape, f"d_input shape mismatch: {d_input.shape}"
print("Test 4 passed: backward gradient has correct shape")

Test 4 passed: backward gradient has correct shape


Test 5: Backward pass gradient values are correct

In [7]:
pool = AvgPool2d(kernel_size=2, stride=2)

x = cp.ones((1, 1, 4, 4), dtype=cp.float32)
pool.forward(x)

# If d_out is all 1s and kernel is 2x2, each input element
# should receive gradient of 1 / (2*2) = 0.25
d_out = cp.ones((1, 1, 2, 2), dtype=cp.float32)
d_input = pool.backward(d_out)

expected = cp.full((1, 1, 4, 4), 0.25, dtype=cp.float32)
cp.testing.assert_allclose(d_input, expected, rtol=1e-5)
print("Test 5 passed: backward gradient values correct (evenly distributed)")

Test 5 passed: backward gradient values correct (evenly distributed)


Test 6: Tuple kernel_size and stride work

In [8]:
pool = AvgPool2d(kernel_size=(2, 3), stride=(1, 2))
x = cp.random.randn(1, 1, 6, 8)
out = pool(x)
# H_out = (6 - 2) / 1 + 1 = 5
# W_out = (8 - 3) / 2 + 1 = 3
assert out.shape == (1, 1, 5, 3), f"Expected (1,1,5,3), got {out.shape}"
print("Test 6 passed: non-square kernel and stride work correctly")

Test 6 passed: non-square kernel and stride work correctly
